In [5]:
import pandas as pd
from pathlib import Path

root = Path("..")  # notebook está em /notebooks
DATA_PROCESSED = root / "data" / "processed"
REPORTS_TABLES = root / "reports" / "tables"

mp = pd.read_parquet(DATA_PROCESSED / "master_proposals.parquet")
mi = pd.read_parquet(DATA_PROCESSED / "master_installments.parquet")

qc = pd.read_csv(REPORTS_TABLES / "modeling_qc_summary.csv")
mr = pd.read_csv(REPORTS_TABLES / "modeling_match_rate_by_parcela.csv")

qc, mr

(                                  check     value
 0                t2_duplicate_unique_id      0.00
 1                           bridge_rows  24660.00
 2                    matched_rows_proxy  14348.00
 3                  match_rate_proxy_pct     58.18
 4     unique_proposals_master_proposals   4961.00
 5  unique_proposals_master_installments   4840.00,
    parcela_n  match_rate_pct
 0          1           66.79
 1          2           63.20
 2          3           59.54
 3          4           56.92
 4          5           51.69
 5          6           46.64)

In [7]:
# Gate 1: chaves
assert qc.loc[qc["check"]=="t2_duplicate_unique_id","value"].iloc[0] == 0, "unique id duplicado na tabela 2!"

# Gate 2: existência das colunas essenciais
required_mp = ["proposal_id", "base_date", "base_month", "is_paid_base"]
required_mi = ["proposal_id", "parcela_n", "payment_unique_id", "base_month"]

missing_mp = [c for c in required_mp if c not in mp.columns]
missing_mi = [c for c in required_mi if c not in mi.columns]

print("Missing master_proposals:", missing_mp)
print("Missing master_installments:", missing_mi)

assert len(missing_mp) == 0, "Faltam colunas essenciais em master_proposals!"
assert len(missing_mi) == 0, "Faltam colunas essenciais em master_installments!"

# Gate 3: base_month saudável
pct_base_month_null = mp["base_month"].isna().mean() * 100
print("Pct base_month nulo:", round(pct_base_month_null, 2), "%")

# Gate 4: parcela_n dentro do esperado
print("parcelas em master_installments:", sorted(mi["parcela_n"].dropna().unique()))

Missing master_proposals: []
Missing master_installments: []
Pct base_month nulo: 0.12 %
parcelas em master_installments: [1, 2, 3, 4, 5, 6]


In [9]:
# Propostas totais e pagas base
n_props = mp["proposal_id"].nunique()
paid_base_rate = mp["is_paid_base"].mean() * 100

print("Propostas únicas:", n_props)
print("Paid base rate (%):", round(paid_base_rate, 2))

# Cobertura por parcela (match proxy já veio no mr, mas aqui vemos o que tem status)
if "Status" in mi.columns:
    status_fill = (mi.groupby("parcela_n")["Status"].apply(lambda s: s.notna().mean()*100)).round(2)
    print("\n% linhas com Status preenchido por parcela:")
    display(status_fill)

    paid_rate_by_parcela = (mi.assign(is_paid=mi["Status"].astype("string").str.lower().eq("pago"))
                            .groupby("parcela_n")["is_paid"].mean()*100).round(2)
    print("\n% Pago por parcela (entre as linhas existentes):")
    display(paid_rate_by_parcela)

# Diagnóstico: propostas que existem no mp mas não têm nenhuma linha no mi
props_in_mi = mi["proposal_id"].nunique()
print("\nPropostas com ao menos 1 parcela linkada:", props_in_mi)
print("Propostas sem parcelas linkadas:", n_props - props_in_mi)

Propostas únicas: 4961
Paid base rate (%): 99.88

% linhas com Status preenchido por parcela:


parcela_n
1     99.91
2    100.00
3     99.98
4    100.00
5    100.00
6     99.97
Name: Status, dtype: float64


% Pago por parcela (entre as linhas existentes):


parcela_n
1    91.41
2    87.22
3     82.9
4    79.03
5    75.13
6    71.81
Name: is_paid, dtype: Float64


Propostas com ao menos 1 parcela linkada: 4840
Propostas sem parcelas linkadas: 121


In [11]:
coverage = (mi.groupby(["base_month","parcela_n"])
            .agg(
                n_rows=("payment_unique_id","size"),
                n_props=("proposal_id","nunique"),
                pct_status_nonnull=("Status", lambda s: s.notna().mean()*100 if "Status" in mi.columns else None),
            )
            .reset_index())

coverage.to_csv(REPORTS_TABLES / "installments_coverage_by_base_month_parcela.csv", index=False)
coverage.head()

,base_month,parcela_n,n_rows,n_props,pct_status_nonnull
0,2024-04,1,3,3,100.0
1,2024-04,2,3,3,100.0
2,2024-04,3,3,3,100.0
3,2024-04,4,3,3,100.0
4,2024-04,5,3,3,100.0
